========================================================================
  CÂU HỎI THỰC HÀNH — U-Net Lane Segmentation
  Notebook dành cho Google Colab (GPU runtime)
========================================================================

CÁCH SỬ DỤNG:
1. Mở Google Colab → chọn Runtime → Change runtime type → GPU (T4)
2. Chạy tất cả cell từ trên xuống
3. Hoàn thiện class Decoder ở CELL TODO
4. Chạy train + inference
5. Đối chiếu ảnh prediction với đáp án

========================================================================


# I. DOWNLOAD DATASET

In [ ]:
!unzip data/Dataset_Lane_Segmentation-main.zip

#II. SETUP & FIXED SEED

In [ ]:
import os
import random
import copy
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Seed = {SEED}")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

#III. DEFINE COLOR MAP

In [ ]:
n_colors = 3
ice_colors = n_colors - 1
jet = plt.get_cmap('jet', ice_colors)
newcolors = jet(np.linspace(0, 1, ice_colors))
black = np.array([[0, 0, 0, 1]])
newcolors = np.concatenate((newcolors, black), axis=0)
cmap = ListedColormap(newcolors)


# IV. READ DATA

In [ ]:
ROOT_PATH = '/content/Dataset_Lane_Segmentation-main'
image_dir = os.path.join(ROOT_PATH, 'train')
mask_dir = os.path.join(ROOT_PATH, 'trainanot')
test_image_dir = os.path.join(ROOT_PATH, 'val')
test_mask_dir = os.path.join(ROOT_PATH, 'valanot')

IMG_SIZE = (80, 160)   # (H, W)
NUM_CLASSES = 3
BATCH_SIZE = 64
NUM_EPOCHS = 5
class LaneDataset(Dataset):
    def __init__(self, image_paths, mask_paths):
        self.image_paths = sorted(image_paths)
        self.mask_paths = sorted(mask_paths)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Đọc ảnh BGR → RGB, resize, normalize [0,1]
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE[1], IMG_SIZE[0]))  # (W, H)
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))  # (C, H, W)

        # Đọc mask grayscale, resize (NEAREST), giữ nguyên class index
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (IMG_SIZE[1], IMG_SIZE[0]),
                          interpolation=cv2.INTER_NEAREST)
        mask = mask.astype(np.int64)

        return torch.from_numpy(img), torch.from_numpy(mask)


# Lấy danh sách ảnh
image_paths = sorted([os.path.join(image_dir, f) for f in os.listdir(image_dir)])
mask_paths = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir)])
test_image_paths = sorted([os.path.join(test_image_dir, f) for f in os.listdir(test_image_dir)])
test_mask_paths = sorted([os.path.join(test_mask_dir, f) for f in os.listdir(test_mask_dir)])

# Split train/val (90:10) rồi lấy 1/4 train
train_imgs_full, val_imgs, train_masks_full, val_masks = train_test_split(
    image_paths, mask_paths, test_size=0.1, random_state=SEED
)
# Giảm dataset xuống 1/4 để train nhanh hơn
train_imgs, _, train_masks, _ = train_test_split(
    train_imgs_full, train_masks_full, test_size=0.75, random_state=SEED
)

train_set = LaneDataset(train_imgs, train_masks)
val_set = LaneDataset(val_imgs, val_masks)
test_set = LaneDataset(test_image_paths, test_mask_paths)

train_loader = DataLoader(
    train_set, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, worker_init_fn=seed_worker, generator=g
)
val_loader = DataLoader(
    val_set, batch_size=BATCH_SIZE, num_workers=2,
    worker_init_fn=seed_worker, generator=g
)
test_loader = DataLoader(
    test_set, batch_size=BATCH_SIZE, num_workers=2,
    worker_init_fn=seed_worker, generator=g
)

print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")
print(f"Image size: {IMG_SIZE}, Classes: {NUM_CLASSES}, Batch: {BATCH_SIZE}")

#V. LOSS FUNCTION

In [ ]:
class SparseFocalLoss(nn.Module):
    """
    Focal Loss cho bài toán segmentation multi-class.
    Giảm weight của các sample dễ, tập trung vào sample khó.
    """
    def __init__(self, alpha=0.25, gamma=2.0, num_classes=3):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.num_classes = num_classes

    def forward(self, y_pred, y_true):
        y_pred_soft = F.softmax(y_pred, dim=1)
        y_true_oh = F.one_hot(y_true, self.num_classes)
        y_true_oh = y_true_oh.permute(0, 3, 1, 2).float()
        y_pred_soft = torch.clamp(y_pred_soft, 1e-7, 1.0 - 1e-7)
        loss = -y_true_oh * torch.log(y_pred_soft)
        loss = self.alpha * torch.pow(1 - y_pred_soft, self.gamma) * loss
        return loss.sum(dim=1).mean()

# %%

#VI. BUILD MODEL

##VI.1 ENCODER

In [ ]:
class ConvBlock(nn.Module):
    """Conv3x3 → BN → ReLU → Conv3x3 → BN → ReLU"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class EncoderBlock(nn.Module):
    """ConvBlock → MaxPool2d(2)
    Trả về: (pool_output, skip_connection)
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = ConvBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        x = self.conv(x)
        return self.pool(x), x    # (pool_output, skip)

##VI.2 DECODER:  TODO — Hoàn thiện DecoderBlock

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Upsample with ConvTranspose2d to double spatial dimensions
        self.upsample = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        # After concatenating skip connection, we have out_channels*2, so reduce back
        self.conv = ConvBlock(out_channels * 2, out_channels)

    def forward(self, x1, x2):
        # Upsample x1 to match spatial dimensions of x2
        x1 = self.upsample(x1)
        # Concatenate along channel dimension (dim=1)
        x = torch.cat([x1, x2], dim=1)
        # Apply ConvBlock to refine features
        x = self.conv(x)
        return x


##VI.3: U-NET MODEL

In [ ]:
class UNet(nn.Module):
    """
    U-Net custom cho Lane Segmentation
    Encoder: 32 → 64 → 128 → 128 → 128  | Bottleneck: 512
    Decoder: 128 →128 → 128 → 64 → 32 | Output: 3 classes
    """
    def __init__(self, in_channels=3, num_classes=3):
        super().__init__()
        # Encoder
        self.enc0 = EncoderBlock(in_channels, 32)
        self.enc1 = EncoderBlock(32, 64)
        self.enc2 = EncoderBlock(64, 128)
        self.enc3 = EncoderBlock(128, 128)
        self.enc4 = EncoderBlock(128, 128)

        # Bottleneck
        self.center = ConvBlock(128, 512)

        # Decoder
        self.dec4 = DecoderBlock(512, 128)
        self.dec3 = DecoderBlock(128, 128)
        self.dec2 = DecoderBlock(128, 128)
        self.dec1 = DecoderBlock(128, 64)
        self.dec0 = DecoderBlock(64, 32)

        # Output head
        self.out_conv = nn.Sequential(
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 1),
        )

    def forward(self, x):
        p0, s0 = self.enc0(x)
        p1, s1 = self.enc1(p0)
        p2, s2 = self.enc2(p1)
        p3, s3 = self.enc3(p2)
        p4, s4 = self.enc4(p3)
        center = self.center(p4)
        d4 = self.dec4(center, s4)
        d3 = self.dec3(d4, s3)
        d2 = self.dec2(d3, s2)
        d1 = self.dec1(d2, s1)
        d0 = self.dec0(d1, s0)

        return self.out_conv(d0)

##VI.4 MODEL INITIALIZATION

In [ ]:

model = UNet(in_channels=3, num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")


#=== Metric — mIoU ===

def compute_mIoU(y_pred, y_true, num_classes=3):
    """Tính Mean Intersection over Union."""
    pred = torch.argmax(y_pred, dim=1).view(-1)
    true = y_true.view(-1)
    ious = []
    for cls in range(num_classes):
        pred_cls = (pred == cls)
        true_cls = (true == cls)
        intersection = (pred_cls & true_cls).sum().float()
        union = (pred_cls | true_cls).sum().float()
        if union > 0:
            ious.append((intersection / union).item())
    return np.mean(ious) if ious else 0.0

#VII. TRAINNING LOOP

In [ ]:
criterion = SparseFocalLoss(alpha=0.25, gamma=2.0, num_classes=NUM_CLASSES)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

best_val_loss = float('inf')
best_model_wts = None

print("=" * 70)
print(f"Training U-Net — {NUM_EPOCHS} epochs, {len(train_set)} images")
print("=" * 70)

for epoch in range(NUM_EPOCHS):
    # --- Train ---
    model.train()
    train_loss = 0.0
    train_miou = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_miou += compute_mIoU(outputs, masks, NUM_CLASSES)

    train_loss /= len(train_loader)
    train_miou /= len(train_loader)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    val_miou = 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            val_loss += loss.item()
            val_miou += compute_mIoU(outputs, masks, NUM_CLASSES)

    val_loss /= len(val_loader)
    val_miou /= len(val_loader)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:2d}/{NUM_EPOCHS}]  "
          f"Train Loss: {train_loss:.4f} | mIoU: {train_miou:.4f}  ||  "
          f"Val Loss: {val_loss:.4f} | mIoU: {val_miou:.4f}")

print("=" * 70)
print(f"Best Val Loss: {best_val_loss:.4f}")
print("Training complete!")